# Model Comparison

Now that we have all of these different models, we need to compare their performance

In [1]:
import sys

sys.path.append("..")

from src.data.load_data import load_all_data

ratings, movies, users = load_all_data()

In [3]:
from src.data.split import train_test_split_by_user

train_df, test_df = train_test_split_by_user(ratings, test_size=0.2)

set(train_df.index).intersection(set(test_df.index))

user = ratings["user_id"].iloc[0]

train_user = train_df[train_df["user_id"] == user]
test_user = test_df[test_df["user_id"] == user]

In [4]:
import pandas as pd

from src.evaluation.evaluate import evaluate_model
from src.features.content import build_genre_matrix
from src.models.popularity import PopularityRecommender
from src.models.content import ContentRecommender
from src.models.item_collaborative import ItemCollaborativeRecommender
from src.models.hybrid import HybridRecommender

genre_matrix = build_genre_matrix(movies)

models = {
    "popularity_bayesian": PopularityRecommender(method="bayesian").fit(train_df),
    "content_based": ContentRecommender().fit(movies, genre_matrix),
    "item_collaborative": ItemCollaborativeRecommender(min_rating=4).fit(train_df),
    "hybrid": HybridRecommender(
        popularity_model=PopularityRecommender(method="bayesian"),
        content_model=ContentRecommender(),
        collaborative_model=ItemCollaborativeRecommender(min_rating=4),
        weights={
            "popularity": 0.15,
            "content": 0.25,
            "collaborative": 0.60,
        },
    ).fit(train_df, movies, genre_matrix),
}

results = []

for name, model in models.items():
    metrics = evaluate_model(model, train_df, test_df, k=10)
    metrics["model"] = name
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df = results_df[
    ["model", "precision_at_k", "recall_at_k", "ndcg_at_k"]
]

results_df.sort_values("ndcg_at_k", ascending=False)

,model,precision_at_k,recall_at_k,ndcg_at_k
2,item_collaborative,0.131871,0.064133,0.147047
3,hybrid,0.128477,0.061890,0.141710
0,popularity_bayesian,0.054404,0.019753,0.058728
1,content_based,0.035546,0.017031,0.037486
